# Solutions · Chapter 02-07 · Showing what you found, without saying more than you found

Attempt each exercise before reading. Where an exercise asks for a judgement rather than a number,
the solution gives one defensible answer and says what makes it defensible - yours may differ in
wording and still be right.

Run the setup cell first; it rebuilds everything the chapter left in memory.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 200
old = rng.normal(47.0, 11.0, n).round(1)
new = rng.normal(41.0, 11.0, n).round(1)

gen = np.random.default_rng(53)
months = 36
rentals = 100 + np.cumsum(gen.normal(2.0, 6.0, months))
permits = 20 + np.cumsum(gen.normal(0.6, 2.5, months))

noise_rng = np.random.default_rng(42)
n_days, n_candidates = 80, 40
candidates = noise_rng.normal(0, 1, (n_days, n_candidates))
satisfaction = noise_rng.normal(0, 1, n_days)


def best_abs_r(matrix, target):
    ms = (matrix - matrix.mean(0)) / matrix.std(0)
    ts = (target - target.mean()) / target.std()
    return np.abs(ms.T @ ts / len(target))


print("setup complete: old, new, rentals, permits, candidates, satisfaction, best_abs_r")

## E1 · Why no algorithm can tell the three worlds apart

Because the three datasets are, as far as any algorithm can see, the same dataset: the same pairs of
numbers with the same correlation and the same shape. An algorithm's input is the table, and the
direction of the arrow is a fact about the *process that produced* the table, not a fact recorded
inside it.

A useful way to hold this: three different mechanisms can generate identical data, so no function of
the data can return three different answers for them. Any method that claimed to would have to
return different outputs for identical inputs.

(There are methods that attempt causal direction from observational data - they work by assuming
something extra about the *shape* of the noise, and they are assumptions, not extractions. That is
module 12 territory. The three worlds here are built to be genuinely symmetric, so nothing recovers
them.)

## E2 · Which number answers "do these move together"

**The changes, -0.011.** The question "do these two things move together" means "when one departs
from its usual behaviour, does the other depart too" - and the month-to-month change is what
measures a departure.

The levels correlation of 0.914 is not wrong; it is a correct answer to a different question:
*"are these two series both high in the same months?"* For any two upward-trending series the answer
is automatically yes, because both are low early and high late. That correlation is largely a
measurement of shared time, and it would still be near 0.9 if one series were bike rentals and the
other were the price of coffee.

The general form: correlating two series over time mostly measures whether they share a trend. Take
the trend out - by differencing - and what remains is the relationship.

## E3 · "The new rack is faster, riders will feel the difference"

**Supported:** "the new rack is faster" - on average, by 5.42 seconds, and that difference is the
right basis for a purchasing decision across a million dockings.

**Not supported:** "riders will feel the difference". The number to quote is that a random new
docking beats a random old one only **64.6%** of the time - so more than a third of riders comparing
one docking to one memory will experience the new rack as no better. Alternatively: **28.5%** of new
dockings were slower than the *average* old docking.

A rewritten sentence that keeps the finding and drops the overclaim: *"The new rack saves about five
seconds per docking on average. Individual dockings vary far more than that, so riders are unlikely
to notice the change - the case for it is the city-wide total, not the individual experience."*

## E4 · Ten independent looks

Each column has a 10% chance of exceeding 0.181 by itself. The chance that **none** of ten does is
`0.9 ** 10`, so the chance that at least one does is `1 - 0.9 ** 10`.

In [ ]:
for k in [1, 5, 10, 20]:
    print("looks: %2d   P(at least one |r| > 0.181) = %.3f" % (k, 1 - 0.9 ** k))

**0.651.** With ten independent looks, a "one-in-ten" result becomes a two-in-three result.

**What it implies for a dashboard with 10 charts.** A dashboard is ten simultaneous looks, refreshed
daily. Something on it will look striking almost every time you open it, and most of what looks
striking is the dashboard doing its arithmetic rather than the business doing something. The
practical consequences:

- Decide in advance which one or two charts drive decisions; the rest are context.
- Judge a spike against how often spikes appear when nothing is happening, not against the flat line
  you imagined.
- Be especially careful when someone screenshots a single panel - the other nine looks travel with
  the number and are invisible in the screenshot.

## E5 · Same bars, different spreads

Both charts show bars at 30 and 34. The strip plots are entirely different:

- **Dataset P (sd = 2):** two tight, clearly separated clouds. Nearly every P-group delivery is
  faster than nearly every Q-group one. A customer would notice.
- **Dataset Q (sd = 15):** two wide clouds sitting almost on top of each other. The 4-minute gap is
  a quarter of one standard deviation, and any individual customer's experience is dominated by
  which day they ordered, not which group they were in.

**"Customers will notice" is supported by P and not by Q.** The bar chart cannot distinguish them,
which is the entire point of the exercise: a difference of means is only interpretable next to the
spread it sits in. That ratio - difference divided by spread - is a standard summary called an
effect size, and it is 2.0 for P and 0.27 for Q.

In [ ]:
for label, sd in [("P", 2.0), ("Q", 15.0)]:
    print("dataset %s: gap 4.0 min, sd %.1f  ->  gap / sd = %.2f" % (label, sd, 4.0 / sd))

## E6 · `overlap_report`

In [ ]:
def overlap_report(a, b, name_a="a", name_b="b"):
    a, b = np.asarray(a), np.asarray(b)
    diff = a.mean() - b.mean()
    beyond = (b > a.mean()).mean()
    wins = (b[:, None] < a[None, :]).mean()
    print(f"{name_a} mean {a.mean():.2f}   {name_b} mean {b.mean():.2f}   difference {diff:.2f}")
    print(f"  share of '{name_b}' beyond '{name_a}' mean".ljust(44) + f": {beyond:.3f}")
    print("  P(random second beats random first)".ljust(44) + f": {wins:.3f}")
    print("  difference / pooled sd".ljust(44)
          + f": {diff / np.sqrt((a.var() + b.var()) / 2):.2f}")


overlap_report(old, new, "old rack", "new rack")

print()
tight = np.random.default_rng(5)
overlap_report(tight.normal(47.0, 1.5, n), tight.normal(41.0, 1.5, n),
               "old (tight)", "new (tight)")

The second pair has **the same 6-second design gap and a spread of 1.5 seconds instead of 11**. Not
one of its new dockings exceeds the old average, and a random new docking wins 99.9% of the time.
Identical bars; opposite conclusions about individuals. The last line of the report - difference over
pooled spread - is the number that separates the two cases in one figure.

## E7 · How the best correlation grows with the number of looks

In [ ]:
rows = []
for k in [5, 20, 40, 100]:
    s = np.random.default_rng(100 + k)
    vals = [best_abs_r(s.normal(0, 1, (80, k)), s.normal(0, 1, 80)).max() for _ in range(1500)]
    rows.append({"candidates": k, "median best |r|": round(float(np.median(vals)), 3),
                 "90th pct": round(float(np.quantile(vals, 0.9)), 3)})

table = pd.DataFrame(rows)
print(table.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(table["candidates"], table["median best |r|"], "o-", color="#0072B2")
ax.set_xscale("log")
ax.set_xlabel("number of candidate columns (log scale)")
ax.set_ylabel("median best |r| under pure noise")
ax.set_title("80 rows, nothing related to anything")
ax.set_ylim(0, 0.35)
plt.tight_layout()
plt.show()

**The shape is logarithmic** - straight on a log x-axis. Going from 5 candidates to 20 costs about as
much as going from 20 to 80.

Two consequences worth carrying:

- Doubling your feature list does not double your false-discovery problem; it adds a roughly constant
  amount. This is mildly reassuring.
- But there is no number of columns at which the effect switches off. Even five candidates put the
  typical best at 0.173 - already the sort of correlation people put on slides.

The honest reading of any screening exercise is a comparison: *my best is 0.25, and the noise
benchmark for this many looks and this many rows is 0.265.* Without the benchmark, 0.25 is just a
number that feels meaningful.

## E8 · Levels versus changes, with a warning

In [ ]:
def trend_check(x, y, name_x="x", name_y="y"):
    x, y = np.asarray(x), np.asarray(y)
    r_lev = np.corrcoef(x, y)[0, 1]
    r_chg = np.corrcoef(np.diff(x), np.diff(y))[0, 1]
    print(f"{name_x} vs {name_y}:  levels {r_lev:+.3f}   changes {r_chg:+.3f}")
    if abs(r_lev - r_chg) > 0.5:
        print("  WARNING: the levels correlation is mostly a shared trend.")
        print("           Report the change correlation, or difference the series first.")


trend_check(rentals, permits, "rentals", "permits")

print()
# A genuinely related pair: the second series is built FROM the first.
h = np.random.default_rng(8)
base = 100 + np.cumsum(h.normal(2.0, 6.0, months))
responds = 30 + 0.35 * base + h.normal(0, 1.2, months)
trend_check(base, responds, "base", "responds")

The unrelated pair triggers the warning: levels **+0.914**, changes **-0.011**, a gap of 0.925.

The genuinely related pair does not: levels **+0.991**, changes **+0.756**. The relationship survives
differencing because it is a relationship rather than a coincidence of direction - when `base` jumps
in a month, `responds` jumps in the same month.

That is the diagnostic in one line: **a real relationship survives differencing; a shared trend does
not.** The 0.5 threshold is a convention with nothing behind it - a prompt to look, not a verdict.

## E9 · Conversions and support tickets on twin axes

Three checks, in order, cheapest first:

1. **Difference both series and re-correlate.** Two years of a growing company means everything
   rises. If the correlation of the monthly changes is near zero, the chart is measuring growth and
   there is nothing else to discuss. This costs one line of code and settles most such charts.
2. **Check the axes.** Twin axes have two independently chosen scales, and the apparent tracking is
   partly a decision someone made. Re-plot both series as percentage change from month one on a
   single axis and see whether the story survives.
3. **Check the denominator.** "Support tickets" is a count, and conversions raise the customer base.
   Tickets *per customer* is the quantity that speaks to hiring; the raw count would rise even if
   every customer needed less support than before.

Only if all three survive is the caption worth discussing - and even then it is an association, so
"we need to hire" needs a forecast of tickets per customer, not a correlation.

## E10 · Vitamin D and respiratory infections

| World | A plausible story |
|---|---|
| **Supplements help** | Vitamin D has a role in immune function; correcting a deficiency reduces susceptibility |
| **Reverse** | People who keep getting infections stop bothering with supplements, or people who feel well maintain their routines - the health state drives the supplementing |
| **Common cause** | People who take supplements are, on average, wealthier, more health-conscious, better nourished, less likely to work in crowded conditions. Every one of those independently reduces infections |

**The design that separates them: a randomised controlled trial** - assign supplements by lottery.
Randomisation cuts every arrow *into* the supplement decision, so wealth, health-consciousness and
prior health can no longer differ systematically between the groups, and reverse causation is
impossible because the assignment came first.

This is exactly 00-04's randomised campaign in a different domain, and worth noticing: the answer to
"how do I separate these three stories" was the same answer there. It is nearly always the same
answer.

## E11 · "11 significant out of 200, so some are real"

**Error one: 10 is what you expect when nothing is real.** A 5% threshold means 5% of unrelated
tests pass it by definition. 200 × 0.05 = 10. Finding 11 is not evidence of anything; it is the
noise benchmark plus one.

**Error two: "at least some of these are real" cannot be attached to any particular one.** Even in a
situation where the count *were* elevated - say 30 hits - that would tell you roughly 20 of the 30
are real, without telling you *which* 20. The 11 named columns are not a shortlist of findings;
they are the columns that happened to win this round.

The repair is to compare the count against its null expectation, and then to check the survivors on
data not used to select them.

In [ ]:
print("tests: 200 at a 5% threshold")
print("expected hits if nothing is related : %.0f" % (200 * 0.05))
print("observed hits                       : 11")

## E12 · The interview answer

> "During an exploratory pass I screened about forty candidate variables against a satisfaction
> score on eighty days of data, and the strongest came back at 0.25 - which looked like a finding.
> Before writing it up I ran the same screen on simulated data where nothing was related, and the
> best of forty columns came back at 0.26 on a typical run, so my result was actually below the
> noise benchmark. It also failed on a fresh sample. I reported it as 'screened forty candidates,
> nothing survived', which was a duller slide than the one I nearly gave. Since then I write down
> how many things I looked at, and I keep a holdout for anything I picked by looking."

Five sentences: what was done, what looked promising, how it was checked, what was reported, what
changed. The last sentence is the one being assessed - the interviewer is asking whether you have a
process, and a candidate who has never been wrong is a candidate who has never checked.

## E13 · The hospital's new wing

1. **Claim:** patients treated in the new wing had a lower death rate than patients in the old wing
   over the period examined.
2. **Evidence:** the two rates, each with its patient count and the date range - "4.1% of 1,240
   versus 6.8% of 1,510, January to December".
3. **What else could produce this:** patients are not assigned to wings at random. The most likely
   explanation by far is that the wings admit different patients - if the old wing takes emergencies,
   or the frailest cases, or a specialty with worse outcomes, it will show a higher death rate with
   identical care. Staffing, admission season and transfers between wings all do the same.
4. **What would change my mind:** compare within severity bands and within specialty; if the gap
   survives inside each band, the case-mix explanation is weakened. If it disappears, the wings were
   never comparable.

**The single most valuable piece of information: how patients are allocated to wings.** Everything
else follows from it - and if the answer is "the sicker ones go to the old wing", the comparison
cannot be repaired by any amount of analysis, only by adjusting for the severity and accepting the
residual doubt.

This is Simpson's paradox from 02-06 with a fatal cost attached, and it is why hospital league tables
are risk-adjusted.

## E14 · For the committee chair

> "We know that the people who open our emails are also the people who ride most. What we do not
> know is which way round that works. It might be that the emails remind people to ride. It might
> equally be that the keenest riders are simply the ones who bother to read anything we send - in
> which case the emails are following their enthusiasm, not creating it. Our figures look exactly
> the same either way, so they cannot separate the two. What would separate them is sending the next
> campaign to a randomly chosen half of our riders and comparing the two halves. That takes one
> month and it would let me answer you properly instead of guessing."

Under 100 words, no jargon, and it ends with a concrete offer rather than a refusal - which is what
makes the difference between sounding careful and sounding obstructive.

## E15 · Four worlds at r = 0.60

The first three worlds rebuild the chapter's demonstration at a lower correlation. The fourth is
**mutual causation**: opens and rides each raise the other, settling at an equilibrium. For a
symmetric feedback with strength `b`, the resulting correlation works out to `2b / (1 + b**2)`, so
`b = 1/3` gives 0.600.

In [ ]:
M = 300
b = 1 / 3

g1 = np.random.default_rng(52)
x_a = g1.normal(0, 1, M)
y_a = 0.60 * x_a + g1.normal(0, 0.80, M)          # opens -> rides

g2 = np.random.default_rng(52)
y_b = g2.normal(0, 1, M)
x_b = 0.60 * y_b + g2.normal(0, 0.80, M)          # rides -> opens

g3 = np.random.default_rng(17)
keen = g3.normal(0, 1, M)
x_c = 0.90 * keen + g3.normal(0, 0.7348, M)       # common cause
y_c = 0.90 * keen + g3.normal(0, 0.7348, M)

g4 = np.random.default_rng(61)                    # each raises the other
e_x, e_y = g4.normal(0, 1, M), g4.normal(0, 1, M)
x_d = (e_x + b * e_y) / (1 - b * b)
y_d = (e_y + b * e_x) / (1 - b * b)

four = {"opens -> rides": (x_a, y_a), "rides -> opens": (x_b, y_b),
        "common cause": (x_c, y_c), "each raises the other": (x_d, y_d)}

for label, (x, y) in four.items():
    print("%-24s r = %.3f" % (label, np.corrcoef(x, y)[0, 1]))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharex=True, sharey=True)
for ax, (label, (x, y)) in zip(axes, four.items()):
    xs = (x - x.mean()) / x.std()
    ys = (y - y.mean()) / y.std()
    ax.scatter(xs, ys, s=12, alpha=0.5, color="#0072B2")
    ax.set_title(f"r = {np.corrcoef(x, y)[0, 1]:.3f}", fontsize=10)
    ax.set_xlabel("opens (standardised)")
axes[0].set_ylabel("rides (standardised)")
plt.tight_layout()
plt.show()

**No, the fourth world is not distinguishable either** - four mechanisms, four correlations within
0.003 of each other, four clouds with nothing to choose between them.

The fourth world matters because it is the one people forget. The usual framing offers three
options - forwards, backwards, or a common cause - and quietly implies that one of them is the
answer. Feedback is a fourth, it is extremely common in anything involving human behaviour, and its
intervention effect is different again:

In [ ]:
effects = pd.DataFrame({
    "world": list(four.keys()),
    "r": [round(float(np.corrcoef(x, y)[0, 1]), 3) for x, y in four.values()],
    "effect of forcing opens up by 1": [0.60, 0.00, 0.00, round(b, 3)],
})
print(effects.to_string(index=False))

Four identical pictures, four different answers to the only question anyone cares about, ranging
from 0.6 to nothing.

**What would it take to tell them apart?** Nothing you can do to this table. In order of strength:

- **Randomise the emails.** Under randomisation the arrow into "opens" is cut, so worlds 2 and 3
  give zero and worlds 1 and 4 give their effect. One month, and it answers the question.
- **Get finer timing.** Individual-level open and ride timestamps distinguish "opened, then rode"
  from "rode, then opened" - and feedback shows up as alternation. This weakens the reverse-causation
  story but never rules out a common cause that moves slowly.
- **Find a natural experiment.** A delivery failure that suppressed emails for one region is a
  randomisation somebody else performed for you.

The general lesson, and the reason this chapter closes the theoretical half of the module: the
strength of a conclusion is set by **how the data came to exist**, not by what you do to it
afterwards. Analysis cannot add evidence that collection did not gather.

## Where to go next

**02-08 · A full exploratory analysis and a written data dictionary** applies all seven chapters of
this module to one real dataset from beginning to end. It is the module's applied chapter and it
produces a document rather than an insight - the thing you hand to whoever inherits the data.

If any of this chapter felt loose, the two worth rereading before moving on are **00-04** (the same
causal material, approached through estimation rather than through identifiability) and **02-06**
(Simpson's paradox, which is the aggregation-level version of the same problem).